# Hidden State Extraction

Extracts hidden states from Llama-3.1-8B-Instruct for all generated conversations.

**Design:**
- Layer: last transformer layer (`hidden_states[-1]`)
- Position: last token of each assistant turn
- One forward pass per conversation; turn positions found by tokenizing prefixes

**Output:**
- `data/representations/hidden_states.npy` — float16, shape (N, 4096)
- `data/representations/metadata.parquet` — one row per (conversation × turn)

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID   = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE     = "cuda:0"  # cuda:0 because CUDA_VISIBLE_DEVICES=4
DTYPE      = torch.bfloat16

CONV_ROOT  = repo_root / "data" / "conversations"
SAVE_DIR   = repo_root / "data" / "representations"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

STATES_PATH   = SAVE_DIR / "hidden_states.npy"
METADATA_PATH = SAVE_DIR / "metadata.parquet"

# Map folder → (framework_label, split_label)
# Add benign folders here once generated (run 01_data_generation.ipynb for each framework)
FOLDERS = {
    #"crescendo_harmful_v2":   ("crescendo",   "harmful"),
    "actorattack_harmful_v2": ("actorattack", "harmful"),
    #"xteaming_harmful_v2":    ("xteaming",    "harmful"),
    # Uncomment as benign data becomes available:
    # "crescendo_benign_v2":   ("crescendo",   "benign"),
    "actorattack_benign_v3": ("actorattack", "benign"),
    # "xteaming_benign_v2":    ("xteaming",    "benign"),
}

print(f"Device: {DEVICE}")
print(f"Output: {SAVE_DIR}")
print(f"Folders to process: {list(FOLDERS.keys())}")

Device: cuda:0
Output: /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/representations
Folders to process: ['actorattack_harmful_v2', 'actorattack_benign_v3']


In [3]:
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()
print("Model loaded.")
print(f"  Layers: {model.config.num_hidden_layers}")
print(f"  Hidden dim: {model.config.hidden_size}")

Loading meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded.
  Layers: 32
  Hidden dim: 4096


In [4]:
def build_context(turns: list[dict], framework: str) -> tuple[list[dict], list[dict]]:
    """
    Build messages list and list of assistant turns to extract at.

    For Crescendo: skips turns where is_refusal=True (logged during generation).
    These turns were rolled back from the target's context, so excluding them
    gives a fresh forward pass that matches generation conditions exactly.

    For other frameworks: all turns used as-is.

    Returns (messages, extract_at) where extract_at entries have
    {turn_idx, n_messages} for prefix-tokenization.
    """
    messages = []
    extract_at = []

    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    for turn_idx in sorted(by_idx):
        pair = by_idx[turn_idx]
        user_turn = pair.get("user")
        asst_turn = pair.get("assistant")
        if not user_turn or not asst_turn:
            continue

        if framework == "crescendo" and asst_turn.get("is_refusal", False):
            continue

        messages.append({"role": "user",      "content": user_turn["content"]})
        messages.append({"role": "assistant", "content": asst_turn["content"]})
        extract_at.append({"turn_idx": turn_idx, "n_messages": len(messages)})

    return messages, extract_at


def find_turn_end_positions(tokenizer, messages: list[dict], extract_at: list[dict]) -> list[int]:
    """
    For each assistant turn in extract_at, find the index of its last token
    in the full tokenized sequence using prefix tokenization.
    """
    positions = []
    for entry in extract_at:
        prefix = messages[:entry["n_messages"]]
        ids = tokenizer.apply_chat_template(
            prefix,
            return_tensors="pt",
            add_generation_prompt=False,
        )
        positions.append(ids.shape[1] - 1)
    return positions


@torch.no_grad()
def extract_hidden_states(model, tokenizer, messages: list[dict], positions: list[int]) -> np.ndarray:
    """
    One forward pass on the full conversation.
    Returns array of shape (len(positions), hidden_dim) in float16.
    """
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=False,
    ).to(model.device)

    outputs = model(input_ids, output_hidden_states=True)
    last_hidden = outputs.hidden_states[-1][0]  # (seq_len, hidden_dim)

    vecs = [last_hidden[pos].cpu().to(torch.float16).numpy() for pos in positions]
    return np.stack(vecs, axis=0)  # (n_turns, hidden_dim)


print("Helpers defined.")

Helpers defined.


In [5]:
# ── Main extraction loop ──────────────────────────────────────────────────────
# Resume support: load already-processed (framework, pair_id, attempt) keys
already_done = set()
if METADATA_PATH.exists():
    existing_meta = pd.read_parquet(METADATA_PATH)
    already_done = set(zip(existing_meta["framework"], existing_meta["pair_id"], existing_meta["attempt"]))
    print(f"Resuming: {len(existing_meta)} rows already extracted ({len(already_done)} conversations)")
    all_states = [np.load(str(STATES_PATH))]
    all_meta   = [existing_meta]
else:
    all_states = []
    all_meta   = []
    print("Starting fresh extraction.")

for folder_name, (framework, split) in FOLDERS.items():
    folder = CONV_ROOT / folder_name
    if not folder.exists():
        print(f"Skipping {folder_name} — folder not found")
        continue

    files = sorted(folder.glob("*.json"))
    print(f"\n{framework}/{split}: {len(files)} files")

    for fpath in tqdm(files, desc=f"{framework}/{split}"):
        conv = json.loads(fpath.read_text())
        pair_id = conv["objective_pair_id"]
        attempt = conv.get("attempt", 1)
        verdict = conv["verdict"]

        if (framework, pair_id, attempt) in already_done:
            continue

        turns = conv.get("turns", [])
        if not turns:
            continue

        try:
            messages, extract_at = build_context(turns, framework)
            if not extract_at:
                continue

            positions = find_turn_end_positions(tokenizer, messages, extract_at)
            states = extract_hidden_states(model, tokenizer, messages, positions)

        except Exception as e:
            tqdm.write(f"  ERROR {fpath.name}: {e}")
            continue

        rows = [
            {
                "framework": framework,
                "split":     split,
                "pair_id":   pair_id,
                "attempt":   attempt,
                "turn_idx":  entry["turn_idx"],
                "verdict":   verdict,
                "label":     1 if split == "harmful" else 0,
                "fname":     fpath.name,
            }
            for entry in extract_at
        ]

        all_states.append(states)
        all_meta.append(pd.DataFrame(rows))
        already_done.add((framework, pair_id, attempt))

print("\nExtraction complete. Saving...")

final_states = np.concatenate(all_states, axis=0).astype(np.float16)
final_meta   = pd.concat(all_meta, ignore_index=True)

np.save(str(STATES_PATH), final_states)
final_meta.to_parquet(METADATA_PATH, index=False)

print(f"Saved {final_states.shape[0]} vectors → {STATES_PATH}")
print(f"  Shape: {final_states.shape}  ({final_states.nbytes / 1e6:.1f} MB)")
print(f"  Metadata: {METADATA_PATH}")
final_meta.groupby(["framework", "split", "verdict"])["turn_idx"].count().rename("n_vectors").reset_index()

Starting fresh extraction.

actorattack/harmful: 1000 files


actorattack/harmful:   0%|          | 0/1000 [00:00<?, ?it/s]


actorattack/benign: 1000 files


actorattack/benign:   0%|          | 0/1000 [00:00<?, ?it/s]


Extraction complete. Saving...
Saved 6000 vectors → /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/representations/hidden_states.npy
  Shape: (6000, 4096)  (49.2 MB)
  Metadata: /home/lisahusieva/multi-turn-rep-eng/multi-turn-rep-eng/data/representations/metadata.parquet


,framework,split,verdict,n_vectors
0,actorattack,harmful,jailbroken,1962
1,actorattack,harmful,near_miss,1536
2,actorattack,harmful,refusal,2502


## Verification

Sanity checks on the extracted states.

In [6]:
states = np.load(str(STATES_PATH))
meta   = pd.read_parquet(METADATA_PATH)

print(f"States shape: {states.shape}")
print(f"Metadata rows: {len(meta)}")
print()
print("Vectors per framework/split/verdict:")
display(meta.groupby(["framework", "split", "verdict"])["turn_idx"].count().rename("n_vectors"))
print()
print("Vectors per turn position (across all frameworks):")
display(meta.groupby("turn_idx")["framework"].count().rename("n_vectors"))
print()
print("State stats (sample of 1000):")
sample = states[np.random.choice(len(states), min(1000, len(states)), replace=False)].astype(np.float32)
print(f"  mean={sample.mean():.4f}  std={sample.std():.4f}  min={sample.min():.4f}  max={sample.max():.4f}")
print(f"  NaNs: {np.isnan(sample).sum()}  Infs: {np.isinf(sample).sum()}")

States shape: (6000, 4096)
Metadata rows: 6000

Vectors per framework/split/verdict:


framework    split    verdict   
actorattack  harmful  jailbroken    1962
                      near_miss     1536
                      refusal       2502
Name: n_vectors, dtype: int64


Vectors per turn position (across all frameworks):


turn_idx
1    1000
2    1000
3    1000
4    1000
5    1000
6    1000
Name: n_vectors, dtype: int64


State stats (sample of 1000):
  mean=-0.0041  std=2.3155  min=-15.5000  max=10.9375
  NaNs: 0  Infs: 0
